[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/17_multi_armed_bandits/notebooks/aplicaciones/04_ab_testing.ipynb)

# Notebook 4: A/B Testing Adaptativo con Bandidos

En este notebook comparamos el enfoque tradicional de **A/B testing** (asignacion uniforme de trafico)
con un enfoque **adaptativo** basado en **Thompson Sampling**.

La idea central es simple: en un A/B test clasico, repartimos el trafico equitativamente entre todas las variantes
durante todo el experimento. Esto significa que seguimos enviando visitantes a variantes malas incluso cuando
ya tenemos evidencia de que son inferiores. Con bandidos, podemos **explorar y explotar simultaneamente**,
redirigiendo trafico hacia las mejores variantes conforme aprendemos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "red": "#E94F37", "orange": "#F28C28", "blue": "#2E86AB",
    "green": "#27AE60", "purple": "#8E44AD", "gray": "#7F8C8D",
}

np.random.seed(42)

## Escenario: Optimizacion de un sitio web

Supongamos que tenemos un sitio web y queremos optimizar la tasa de conversion (porcentaje de visitantes
que completan una accion deseada, como registrarse o comprar). Tenemos **5 variantes** del sitio:

| Variante | Descripcion | Tasa de conversion real |
|----------|-------------|------------------------|
| Original | Diseno actual | 2% |
| Color | Cambio de colores del boton | 3% |
| Layout | Reorganizacion del contenido | 5% |
| Copy | Texto persuasivo mejorado | 6% |
| Full Redesign | Rediseno completo | 8% |

Tenemos un presupuesto de **T = 10,000 visitantes**. Cada visitante que no convierte es una oportunidad perdida.

In [ ]:
# Scenario setup
conversion_rates = np.array([0.02, 0.03, 0.05, 0.06, 0.08])
variant_names = ["Original", "Color", "Layout", "Copy", "Full Redesign"]
K = len(conversion_rates)
T = 10_000  # total visitors

VARIANT_COLORS = [
    COLORS["gray"], COLORS["orange"], COLORS["blue"],
    COLORS["green"], COLORS["purple"],
]

print(f"Variantes: {K}")
print(f"Visitantes totales: {T:,}")
print(f"Mejor tasa de conversion: {conversion_rates.max():.0%} ({variant_names[np.argmax(conversion_rates)]})")

## A/B Test Tradicional

En un A/B test clasico, dividimos el trafico **equitativamente** entre las 5 variantes.
Cada variante recibe $T/K = 2{,}000$ visitantes. Al final del experimento, comparamos
las tasas de conversion observadas y elegimos la mejor.

El problema: estamos "desperdiciando" visitantes en variantes inferiores durante **todo** el experimento.

In [ ]:
# Traditional A/B test: equal allocation
visitors_per_variant = T // K

ab_conversions = np.array([
    np.random.binomial(visitors_per_variant, p) for p in conversion_rates
])
ab_rates = ab_conversions / visitors_per_variant
ab_total_conversions = ab_conversions.sum()

print("=" * 55)
print(f"{'Variante':<16} {'Visitantes':>10} {'Conversiones':>13} {'Tasa':>8}")
print("=" * 55)
for i in range(K):
    print(f"{variant_names[i]:<16} {visitors_per_variant:>10,} {ab_conversions[i]:>13,} {ab_rates[i]:>8.2%}")
print("-" * 55)
print(f"{'TOTAL':<16} {T:>10,} {ab_total_conversions:>13,} {ab_total_conversions/T:>8.2%}")
print("=" * 55)

## Thompson Sampling Adaptativo

Ahora usamos **Thompson Sampling** con priors $\text{Beta}(1,1)$ (uniforme) para cada variante.
En cada paso:
1. Muestreamos $\theta_k \sim \text{Beta}(\alpha_k, \beta_k)$ para cada variante $k$
2. Asignamos el visitante a la variante con mayor $\theta_k$
3. Observamos si convierte o no y actualizamos los parametros

Esto naturalmente redirige trafico hacia las variantes con mejor rendimiento.

In [ ]:
# Thompson Sampling simulation
alpha = np.ones(K)  # Beta prior: successes + 1
beta_param = np.ones(K)  # Beta prior: failures + 1

ts_allocations = np.zeros(K, dtype=int)
ts_conversions_count = np.zeros(K, dtype=int)

# Track allocation fractions over time
allocation_history = np.zeros((T, K))

for t in range(T):
    # Sample from posterior
    theta_samples = np.array([
        np.random.beta(alpha[k], beta_param[k]) for k in range(K)
    ])
    # Choose variant with highest sample
    chosen = np.argmax(theta_samples)
    ts_allocations[chosen] += 1

    # Simulate outcome
    reward = np.random.binomial(1, conversion_rates[chosen])
    ts_conversions_count[chosen] += reward

    # Update posterior
    alpha[chosen] += reward
    beta_param[chosen] += 1 - reward

    # Record cumulative allocation fractions
    allocation_history[t] = ts_allocations / (t + 1)

ts_total_conversions = ts_conversions_count.sum()

print("=" * 55)
print(f"{'Variante':<16} {'Visitantes':>10} {'Conversiones':>13} {'Tasa':>8}")
print("=" * 55)
for i in range(K):
    rate = ts_conversions_count[i] / ts_allocations[i] if ts_allocations[i] > 0 else 0
    print(f"{variant_names[i]:<16} {ts_allocations[i]:>10,} {ts_conversions_count[i]:>13,} {rate:>8.2%}")
print("-" * 55)
print(f"{'TOTAL':<16} {T:>10,} {ts_total_conversions:>13,} {ts_total_conversions/T:>8.2%}")
print("=" * 55)

## Comparacion Visual de Asignacion

Comparemos como cada enfoque distribuyo el trafico entre las variantes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Traditional A/B: equal allocation
bars1 = axes[0].bar(variant_names, [visitors_per_variant] * K,
                     color=VARIANT_COLORS, edgecolor="white", linewidth=0.8)
axes[0].set_title(f"A/B Tradicional\nConversiones totales: {ab_total_conversions:,}",
                   fontsize=13, fontweight="bold")
axes[0].set_ylabel("Visitantes asignados")
axes[0].set_ylim(0, T * 1.05)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                 f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9)

# Thompson Sampling: adaptive allocation
bars2 = axes[1].bar(variant_names, ts_allocations,
                     color=VARIANT_COLORS, edgecolor="white", linewidth=0.8)
axes[1].set_title(f"Thompson Sampling\nConversiones totales: {ts_total_conversions:,}",
                   fontsize=13, fontweight="bold")
axes[1].set_ylabel("Visitantes asignados")
axes[1].set_ylim(0, T * 1.05)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                 f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## Conversiones Perdidas

Podemos cuantificar el **costo de oportunidad** del A/B test tradicional.
Cada visitante enviado a una variante suboptima podria haber ido a la mejor variante.
La diferencia en conversiones totales nos dice cuantas conversiones "perdimos" por usar
asignacion uniforme.

In [ ]:
# Lost conversions analysis
best_rate = conversion_rates.max()

# Theoretical maximum: all visitors go to the best variant
theoretical_max = T * best_rate

# Expected conversions under each strategy
expected_ab = sum(visitors_per_variant * p for p in conversion_rates)

lost_ab = theoretical_max - ab_total_conversions
lost_ts = theoretical_max - ts_total_conversions
saved = ab_total_conversions - ts_total_conversions  # negative means TS is better

print(f"Maximo teorico (todos a la mejor variante): {theoretical_max:,.0f} conversiones")
print(f"")
print(f"A/B tradicional:     {ab_total_conversions:>6,} conversiones ({lost_ab:,.0f} perdidas vs. maximo)")
print(f"Thompson Sampling:   {ts_total_conversions:>6,} conversiones ({lost_ts:,.0f} perdidas vs. maximo)")
print(f"")
print(f"Thompson Sampling obtuvo {ts_total_conversions - ab_total_conversions:,} conversiones adicionales")
print(f"Mejora relativa: {(ts_total_conversions - ab_total_conversions) / ab_total_conversions:.1%}")

## Evolucion de la Asignacion de Trafico

El siguiente grafico muestra como Thompson Sampling redistribuye el trafico a lo largo del tiempo.
Al inicio, el algoritmo explora todas las variantes. Conforme acumula evidencia,
concentra el trafico en la variante con mayor tasa de conversion.

In [ ]:
# Stacked area chart: allocation evolution
fig, ax = plt.subplots(figsize=(12, 6))

# Subsample for smoother plot
step = 10
x = np.arange(0, T, step)
y = allocation_history[::step].T  # shape: (K, T/step)

ax.stackplot(x, y, labels=variant_names, colors=VARIANT_COLORS, alpha=0.85)

ax.set_xlabel("Visitante")
ax.set_ylabel("Fraccion del trafico")
ax.set_title("Evolucion de la asignacion de trafico (Thompson Sampling)",
              fontsize=13, fontweight="bold")
ax.set_xlim(0, T)
ax.set_ylim(0, 1)
ax.legend(loc="center right", fontsize=9)

plt.tight_layout()
plt.show()

## Analisis de Tamano de Muestra

En un A/B test tradicional, necesitamos calcular el **tamano de muestra** necesario para
detectar una diferencia estadisticamente significativa entre dos variantes.

Pregunta: si queremos distinguir entre la variante "Copy" ($p_1 = 0.06$) y "Full Redesign" ($p_2 = 0.08$),
con un nivel de significancia $\alpha = 0.05$ y poder estadistico $1 - \beta = 0.80$,
cuantos visitantes necesitamos **por variante**?

Usamos la formula clasica para dos proporciones:

$$n = \left(\frac{z_{\alpha/2} \sqrt{2\bar{p}(1-\bar{p})} + z_{\beta}\sqrt{p_1(1-p_1) + p_2(1-p_2)}}{p_2 - p_1}\right)^2$$

donde $\bar{p} = (p_1 + p_2)/2$.

In [ ]:
# Sample size calculation for traditional A/B test
p1, p2 = 0.06, 0.08
alpha_sig = 0.05
power = 0.80

p_bar = (p1 + p2) / 2
z_alpha = stats.norm.ppf(1 - alpha_sig / 2)
z_beta = stats.norm.ppf(power)

numerator = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar))
             + z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2)))
n_required = (numerator / (p2 - p1)) ** 2
n_required = int(np.ceil(n_required))

print(f"Para detectar diferencia entre p1={p1:.0%} y p2={p2:.0%}:")
print(f"  Nivel de significancia: {alpha_sig}")
print(f"  Poder estadistico: {power}")
print(f"  Tamano de muestra requerido por variante: {n_required:,}")
print(f"  Total de visitantes (2 variantes): {2 * n_required:,}")
print()

# How quickly does Thompson identify the best?
# Find the first time the best variant has >50% of allocation
best_idx = np.argmax(conversion_rates)
threshold = 0.50
convergence_point = T  # default if never reached
for t in range(100, T):
    if allocation_history[t, best_idx] >= threshold:
        convergence_point = t
        break

print(f"Thompson Sampling asigna >{threshold:.0%} del trafico a '{variant_names[best_idx]}' "
      f"despues de {convergence_point:,} visitantes.")
print(f"El A/B test tradicional necesita {2 * n_required:,} visitantes solo para comparar 2 variantes.")

## Ejercicio: Disena tu propio experimento

Ahora es tu turno. Define tu propio escenario de A/B testing:
- Elige un contexto (app movil, emails, precios, etc.)
- Define las variantes y sus tasas de conversion esperadas
- Ejecuta ambos enfoques y compara resultados

Modifica los valores marcados con `# <-- CHANGE THIS`.

In [ ]:
# === YOUR EXPERIMENT ===
np.random.seed(123)

my_variant_names = ["Control", "Variante A", "Variante B"]  # <-- CHANGE THIS
my_rates = [0.10, 0.12, 0.15]  # <-- CHANGE THIS
my_T = 5_000  # <-- CHANGE THIS

my_K = len(my_rates)
my_rates = np.array(my_rates)
my_colors = [COLORS["gray"], COLORS["blue"], COLORS["red"]]  # <-- CHANGE THIS (match length)

# --- Traditional A/B ---
my_visitors_each = my_T // my_K
my_ab_conv = np.array([np.random.binomial(my_visitors_each, p) for p in my_rates])
my_ab_total = my_ab_conv.sum()

# --- Thompson Sampling ---
my_alpha = np.ones(my_K)
my_beta = np.ones(my_K)
my_ts_alloc = np.zeros(my_K, dtype=int)
my_ts_conv = np.zeros(my_K, dtype=int)

for t in range(my_T):
    theta = np.array([np.random.beta(my_alpha[k], my_beta[k]) for k in range(my_K)])
    chosen = np.argmax(theta)
    my_ts_alloc[chosen] += 1
    reward = np.random.binomial(1, my_rates[chosen])
    my_ts_conv[chosen] += reward
    my_alpha[chosen] += reward
    my_beta[chosen] += 1 - reward

my_ts_total = my_ts_conv.sum()

# --- Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(my_variant_names, [my_visitors_each] * my_K, color=my_colors, edgecolor="white")
axes[0].set_title(f"A/B Tradicional\nConversiones: {my_ab_total:,}", fontweight="bold")
axes[0].set_ylabel("Visitantes")

axes[1].bar(my_variant_names, my_ts_alloc, color=my_colors, edgecolor="white")
axes[1].set_title(f"Thompson Sampling\nConversiones: {my_ts_total:,}", fontweight="bold")
axes[1].set_ylabel("Visitantes")

plt.tight_layout()
plt.show()

print(f"\nThompson Sampling obtuvo {my_ts_total - my_ab_total:,} conversiones adicionales.")

## Reflexion

Considera las siguientes preguntas:

1. **Cuando preferirias un A/B test tradicional?**
   - Cuando necesitas **validez estadistica rigurosa** (p-values, intervalos de confianza formales).
   - En entornos regulados (farmaceutica, finanzas) donde se requiere un protocolo pre-registrado.
   - Cuando el tamano del efecto es muy pequeno y necesitas control estricto del error tipo I.

2. **Cuando preferirias Thompson Sampling (adaptativo)?**
   - Cuando el costo de asignar trafico a variantes malas es alto (cada conversion perdida es dinero).
   - Cuando tienes muchas variantes y explorar todas equitativamente es prohibitivo.
   - En entornos de produccion donde quieres optimizar mientras aprendes.

3. **Limitaciones del enfoque adaptativo:**
   - Los p-values clasicos no son directamente aplicables (el muestreo no es i.i.d.).
   - Puede converger prematuramente si las diferencias son pequenas.
   - Requiere implementacion en tiempo real, no basta con analizar datos al final.

La eleccion entre ambos enfoques depende del **contexto**, los **costos** y los **requisitos estadisticos** del problema.